# 01 - Extracao: SQL Server para MinIO (CSV)

Este notebook le as quatro tabelas do banco **LojaDB** no SQL Server usando **Spark/JDBC** e grava um arquivo CSV por tabela no bucket **landing-zone** do MinIO.

Fluxo executado:

`SQL Server / LojaDB -> clientes, produtos, pedidos, itens_pedido -> MinIO / landing-zone`

## 1. Configuracao

In [1]:
import os
import shutil
import tempfile
from pathlib import Path

import boto3
from botocore.client import Config
from dotenv import load_dotenv
from pyspark.sql import SparkSession

# Configurar JAVA_HOME e reduzir warnings
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-11-openjdk-amd64'
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

load_dotenv(override=True)

DB_SERVER = os.getenv('DB_SERVER')
DB_PORT = os.getenv('DB_PORT')
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_DATABASE = os.getenv('DB_DATABASE', 'LojaDB')

MINIO_ENDPOINT = os.getenv('MINIO_ENDPOINT')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY')
LANDING_BUCKET = os.getenv('MINIO_LANDING_BUCKET', 'landing-zone')

tabelas = ['clientes', 'produtos', 'pedidos', 'itens_pedido']

print(f'SQL Server: {DB_SERVER}:{DB_PORT}/{DB_DATABASE}')
print(f'MinIO: {MINIO_ENDPOINT} | Bucket: {LANDING_BUCKET}')
print(f'Tabelas: {tabelas}')
print(f'JAVA_HOME: {os.environ.get("JAVA_HOME")}')

SQL Server: localhost:1433/LojaDB
MinIO: http://localhost:9020 | Bucket: landing-zone
Tabelas: ['clientes', 'produtos', 'pedidos', 'itens_pedido']
JAVA_HOME: /usr/lib/jvm/java-11-openjdk-amd64


## 2. Criar SparkSession com driver JDBC do SQL Server

In [2]:
spark = (
    SparkSession.builder
    .appName('SQLServer_to_MinIO_CSV')
    .master('local[*]')
    .config('spark.jars.packages', 'com.microsoft.sqlserver:mssql-jdbc:12.8.1.jre11')
    .config('spark.driver.host', '127.0.0.1')
    .config('spark.driver.bindAddress', '127.0.0.1')
    .config('spark.ui.showConsoleProgress', 'false')
    .getOrCreate()
)

spark.sparkContext.setLogLevel('ERROR')
print('SparkSession criada com sucesso.')

:: loading settings :: url = jar:file:/home/anderson/git-clone/spark-delta-minio/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/anderson/.ivy2/cache
The jars for the packages stored in: /home/anderson/.ivy2/jars
com.microsoft.sqlserver#mssql-jdbc added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-84c33094-549a-49e6-b4cf-8e7987006602;1.0
	confs: [default]
	found com.microsoft.sqlserver#mssql-jdbc;12.8.1.jre11 in central
:: resolution report :: resolve 140ms :: artifacts dl 3ms
	:: modules in use:
	com.microsoft.sqlserver#mssql-jdbc;12.8.1.jre11 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   0   |   0   |   0   ||   1   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-84c

SparkSession criada com sucesso.


## 3. Configurar conexao JDBC

In [3]:
jdbc_url = (
    f'jdbc:sqlserver://{DB_SERVER}:{DB_PORT};'
    f'databaseName={DB_DATABASE};'
    'encrypt=true;'
    'trustServerCertificate=true;'
)

jdbc_properties = {
    'user': DB_USER,
    'password': DB_PASSWORD,
    'driver': 'com.microsoft.sqlserver.jdbc.SQLServerDriver'
}

print(jdbc_url)

jdbc:sqlserver://localhost:1433;databaseName=LojaDB;encrypt=true;trustServerCertificate=true;


## 4. Criar e limpar bucket landing-zone

In [4]:
s3_client = boto3.client(
    's3',
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

try:
    s3_client.head_bucket(Bucket=LANDING_BUCKET)
    print(f'Bucket [{LANDING_BUCKET}] ja existe.')
except Exception:
    s3_client.create_bucket(Bucket=LANDING_BUCKET)
    print(f'Bucket [{LANDING_BUCKET}] criado.')

response = s3_client.list_objects_v2(Bucket=LANDING_BUCKET)
objetos = [{'Key': obj['Key']} for obj in response.get('Contents', [])]

if objetos:
    s3_client.delete_objects(Bucket=LANDING_BUCKET, Delete={'Objects': objetos})
    print(f'{len(objetos)} objeto(s) removido(s) do bucket [{LANDING_BUCKET}].')
else:
    print(f'Bucket [{LANDING_BUCKET}] ja estava vazio.')

Bucket [landing-zone] ja existe.
4 objeto(s) removido(s) do bucket [landing-zone].


## 5. Extrair tabelas do SQL Server e enviar CSVs para o MinIO

In [5]:
resultados = []
tmp_root = Path(tempfile.mkdtemp(prefix='lojadb_csv_'))

try:
    for tabela in tabelas:
        print(f'Extraindo tabela dbo.{tabela}...')

        df = spark.read.jdbc(
            url=jdbc_url,
            table=f'dbo.{tabela}',
            properties=jdbc_properties
        )

        registros = df.count()
        output_dir = tmp_root / tabela

        (
            df.coalesce(1)
            .write
            .mode('overwrite')
            .option('header', True)
            .csv(str(output_dir))
        )

        part_file = next(output_dir.glob('part-*.csv'))
        s3_key = f'{tabela}.csv'

        s3_client.upload_file(
            Filename=str(part_file),
            Bucket=LANDING_BUCKET,
            Key=s3_key,
            ExtraArgs={'ContentType': 'text/csv'}
        )

        tamanho_kb = part_file.stat().st_size / 1024
        resultados.append({
            'tabela': tabela,
            'arquivo': s3_key,
            'registros': registros,
            'colunas': len(df.columns),
            'tamanho_kb': round(tamanho_kb, 1)
        })

        print(f'  {s3_key} enviado para s3://{LANDING_BUCKET}/{s3_key} ({registros} registros, {tamanho_kb:.1f} KB)')
finally:
    shutil.rmtree(tmp_root, ignore_errors=True)

print('Extracao concluida.')

Extraindo tabela dbo.clientes...
  clientes.csv enviado para s3://landing-zone/clientes.csv (100 registros, 7.5 KB)
Extraindo tabela dbo.produtos...
  produtos.csv enviado para s3://landing-zone/produtos.csv (50 registros, 2.5 KB)
Extraindo tabela dbo.pedidos...
  pedidos.csv enviado para s3://landing-zone/pedidos.csv (200 registros, 6.2 KB)
Extraindo tabela dbo.itens_pedido...
  itens_pedido.csv enviado para s3://landing-zone/itens_pedido.csv (400 registros, 6.9 KB)
Extracao concluida.


## 6. Validacao do bucket landing-zone

In [6]:
response = s3_client.list_objects_v2(Bucket=LANDING_BUCKET)
objetos = sorted(response.get('Contents', []), key=lambda obj: obj['Key'])

print(f'Arquivos no bucket [{LANDING_BUCKET}]:\n')
for obj in objetos:
    print(f'  {obj["Key"]:<25} {obj["Size"] / 1024:>8.1f} KB')

print('\nResumo da extracao:')
for item in resultados:
    print(f'  {item["tabela"]:<15} {item["registros"]:>6} registros | {item["colunas"]:>2} colunas | {item["tamanho_kb"]:>6.1f} KB')

Arquivos no bucket [landing-zone]:

  clientes.csv                   7.5 KB
  itens_pedido.csv               6.9 KB
  pedidos.csv                    6.2 KB
  produtos.csv                   2.5 KB

Resumo da extracao:
  clientes           100 registros |  7 colunas |    7.5 KB
  produtos            50 registros |  6 colunas |    2.5 KB
  pedidos            200 registros |  5 colunas |    6.2 KB
  itens_pedido       400 registros |  5 colunas |    6.9 KB


## 7. Encerrar Spark

In [7]:
spark.stop()
print('SparkSession finalizada.')

SparkSession finalizada.
